In [1]:
import numpy as np
import pandas as pd
import re

df = pd.read_csv('../Database SSDC 2026/Database SSDC 2026 UNZIP/tracking_company.csv')

# UNIVERSAL

## 2.1. Bentuk dan Struktur

### Jumlah baris dan kolom

In [2]:
print(f"Jumlah baris : {df.shape[0]}")
print(f"Jumlah kolom : {df.shape[1]}")

expected_columns = 13
print(f"Sesuai dokumentasi ({expected_columns})? = ", df.shape[1] == expected_columns)

Jumlah baris : 12000
Jumlah kolom : 13
Sesuai dokumentasi (13)? =  True


In [3]:
df.head(5)

,id_tracking_company,id_talent_req,id_company,nama_perusahaan,posisi,jenis_penempatan,bidang_studi_dicari,progress,request_date,send_date,jumlah_permintaan,jumlah_dikirimkan,list_nim
0,TC001,TR001,C171,PT Prima Data,Project Coordinator Intern,Magang,"Teknik Industri, Manajemen",Shortlisted,14/05/2023,23/05/2023,3,5,"202018732,202312115,20231598,202117594,202224191"
1,TC002,TR002,C638,PT Sentosa Teknologi,Quality Control Staff,Magang,"Teknik Mesin, Farmasi",Closed,07/05/2023,26/05/2023,5,6,"20212739,202212917,202215216,20218748,20232287..."
2,TC003,TR003,C900,CV Sejahtera Systems,Data Analyst,Part-time,"Manajemen, Statistika",Closed,05/03/2023,17/03/2023,2,3,"202320283,202212715,20239389"
3,TC004,TR004,C696,PT Global Nusantara,Mechanical Engineer Intern,Part-time,Teknik Mesin,Shortlisted,26/04/2023,09/05/2023,2,5,"20219529,202213013,202311443,202212086,202315402"
4,TC005,TR005,C506,PT Techno Bersama,Supply Chain Analyst,Magang,"Teknik Industri, Manajemen",On Review,03/06/2023,13/06/2023,1,3,"202111330,20230612,202211633"


### Kolom tak terduga dan hilang

In [4]:
kolom_aktual = set(df.columns)
kolom_dokumentasi = {"id_tracking_company", "id_talent_req", "id_company", 'nama_perusahaan', 'posisi', 'jenis_penempatan', 'bidang_studi_dicari', 'progress', 'request_date', 'send_date', 'jumlah_permintaan', 'jumlah_dikirimkan', 'list_nim'}

tidak_terduga = kolom_aktual - kolom_dokumentasi
hilang = kolom_dokumentasi - kolom_aktual

print("Kolom tak terduga (ada di data, tidak ada di dokumentasi):", tidak_terduga)
print("Kolom hilang (ada di dokumentasi, tidak ada di data)     :", hilang)

Kolom tak terduga (ada di data, tidak ada di dokumentasi): set()
Kolom hilang (ada di dokumentasi, tidak ada di data)     : set()


### Header

In [7]:
preview = pd.read_csv("../Database SSDC 2026/Database SSDC 2026 UNZIP/tracking_company.csv", header=None, nrows=5)
print(preview)

# kalau ternyata ada baris judul/kosong di atas header asli, load ulang dengan skiprows:
# df = pd.read_csv("nama_file.csv", skiprows=1)

                    0              1           2                     3   \
0  id_tracking_company  id_talent_req  id_company       nama_perusahaan   
1                TC001          TR001        C171         PT Prima Data   
2                TC002          TR002        C638  PT Sentosa Teknologi   
3                TC003          TR003        C900  CV Sejahtera Systems   
4                TC004          TR004        C696   PT Global Nusantara   

                           4                 5                           6   \
0                      posisi  jenis_penempatan         bidang_studi_dicari   
1  Project Coordinator Intern            Magang  Teknik Industri, Manajemen   
2       Quality Control Staff            Magang       Teknik Mesin, Farmasi   
3                Data Analyst         Part-time       Manajemen, Statistika   
4  Mechanical Engineer Intern         Part-time                Teknik Mesin   

            7             8           9                  10  \
0     progr

## 2.2. Tipe data

### Tipe data pandas

In [8]:
df.dtypes

id_tracking_company      str
id_talent_req            str
id_company               str
nama_perusahaan          str
posisi                   str
jenis_penempatan         str
bidang_studi_dicari      str
progress                 str
request_date             str
send_date                str
jumlah_permintaan      int64
jumlah_dikirimkan      int64
list_nim                 str
dtype: object

### Numerik vs Object

aman karena list_nim emang teks karena banyak berupa berupa kumpulan string

In [9]:
for kolom in df.select_dtypes(include="object").columns:
    sampel = df[kolom].dropna().astype(str)
    bersih = sampel.str.replace(",", "", regex=False).str.replace(".", "", regex=False).str.replace("-", "", regex=False).str.strip()
    persen_numerik = (bersih.str.isnumeric().sum() / len(sampel) * 100) if len(sampel) > 0 else 0
    if persen_numerik > 50:
        print(f"Kolom '{kolom}': {persen_numerik:.0f}% nilainya terlihat numerik, tapi dtype-nya object")
        print("Contoh nilai unik:", sampel.unique()[:10])

Kolom 'list_nim': 100% nilainya terlihat numerik, tapi dtype-nya object
Contoh nilai unik: <StringArray>
[         '202018732,202312115,20231598,202117594,202224191',
 '20212739,202212917,202215216,20218748,202322870,202320109',
                              '202320283,202212715,20239389',
          '20219529,202213013,202311443,202212086,202315402',
                              '202111330,20230612,202211633',
                                                 '202219230',
                     '202222850,202022016,20218434,20235562',
                                                 '202312520',
                    '202118645,20228196,202121547,201921579',
                             '202221980,202023380,202019983']
Length: 10, dtype: str


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14460\4097874488.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for kolom in df.select_dtypes(include="object").columns:


### Parse date

2023-2025 --> AMAN

In [10]:
kolom_tanggal = ["request_date", "send_date"] 

for kolom in kolom_tanggal:
    parsed = pd.to_datetime(df[kolom], errors="coerce")
    gagal_parse = parsed.isna().sum() - df[kolom].isna().sum()
    print(f"Kolom '{kolom}':")
    print(f"  Gagal di-parse jadi tanggal : {gagal_parse}")
    if parsed.notna().any():
        print(f"  Range tanggal               : {parsed.min()}  s/d  {parsed.max()}")

Kolom 'request_date':
  Gagal di-parse jadi tanggal : 0
  Range tanggal               : 2023-02-01 00:00:00  s/d  2025-01-31 00:00:00
Kolom 'send_date':
  Gagal di-parse jadi tanggal : 0
  Range tanggal               : 2023-02-04 00:00:00  s/d  2025-02-21 00:00:00


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14460\937839243.py:4: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  parsed = pd.to_datetime(df[kolom], errors="coerce")
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14460\937839243.py:4: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  parsed = pd.to_datetime(df[kolom], errors="coerce")


## 2.3. Missing Values

### Persentase isnull

Aman karena draft berarti belum dikirimkan
berarti send_date kosong -->> belum ada pengiriman (karena masih on proses)
berarti list_nim kosong -->> belum ada mahasiswa yang mengirimkan (karena on proses)

In [11]:
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(null_pct[null_pct > 0].sort_values(ascending=False))

send_date    4.98
list_nim     4.98
dtype: float64


In [12]:
sama_baris = df[df["send_date"].isnull() & df["list_nim"].isnull()]
print(f"Baris yang null di KEDUANYA: {len(sama_baris)}")
print(f"Baris null di send_date saja: {df['send_date'].isnull().sum()}")
print(f"Baris null di list_nim saja : {df['list_nim'].isnull().sum()}")

Baris yang null di KEDUANYA: 598
Baris null di send_date saja: 598
Baris null di list_nim saja : 598


In [13]:
print(df[df["send_date"].isnull()]["progress"].value_counts())

progress
Draft    598
Name: count, dtype: int64


In [14]:
print(df[df["list_nim"].isnull()]["progress"].value_counts())

progress
Draft    598
Name: count, dtype: int64


### Null tersamar

-->> tidak ada null tersamar dan tidak ada double information (penulisan) value

In [15]:
print(df.nunique(dropna=False))

id_tracking_company    12000
id_talent_req          12000
id_company              1495
nama_perusahaan         1495
posisi                    74
jenis_penempatan           3
bidang_studi_dicari      113
progress                   5
request_date             731
send_date                750
jumlah_permintaan          5
jumlah_dikirimkan          9
list_nim               11358
dtype: int64


In [16]:
token_null_tersamar = ["", " ", "-", "N/A", "NA", "na", "null", "None", "TBD",
                        "Belum ada", "belum ada", "?", "--"]

kolom_high_cardinality = ["id_tracking_company", "id_talent_req", "id_company",
                           "nama_perusahaan", "list_nim"]

for kolom in kolom_high_cardinality:
    ditemukan = df[kolom].astype(str).str.strip().isin(token_null_tersamar)
    if ditemukan.sum() > 0:
        print(f"Kolom '{kolom}': {ditemukan.sum()} baris null tersamar")
        print(df.loc[ditemukan, kolom].value_counts())
    else:
        print(f"Kolom '{kolom}': aman, tidak ditemukan null tersamar")

Kolom 'id_tracking_company': aman, tidak ditemukan null tersamar
Kolom 'id_talent_req': aman, tidak ditemukan null tersamar
Kolom 'id_company': aman, tidak ditemukan null tersamar
Kolom 'nama_perusahaan': aman, tidak ditemukan null tersamar
Kolom 'list_nim': aman, tidak ditemukan null tersamar


In [17]:
for kolom in df.columns:
    n_unique = df[kolom].nunique(dropna=False)
    if n_unique <= 150:
        print(f"\n{'='*50}")
        print(f"Kolom: '{kolom}'  ({n_unique} distinct values)")
        print(df[kolom].value_counts(dropna=False))
    else:
        print(f"\nKolom '{kolom}' dilewati ({n_unique} distinct values, terlalu banyak untuk dilihat manual)")


Kolom 'id_tracking_company' dilewati (12000 distinct values, terlalu banyak untuk dilihat manual)

Kolom 'id_talent_req' dilewati (12000 distinct values, terlalu banyak untuk dilihat manual)

Kolom 'id_company' dilewati (1495 distinct values, terlalu banyak untuk dilihat manual)

Kolom 'nama_perusahaan' dilewati (1495 distinct values, terlalu banyak untuk dilihat manual)

Kolom: 'posisi'  (74 distinct values)
posisi
Data Analyst                  1393
IT Support                     937
Quality Control Staff          406
Mechanical Engineer Intern     348
Business Analyst               345
                              ... 
Mobile Developer                62
Data Science Intern             57
Software Engineer Intern        53
DevOps Engineer                 52
QA Engineer                     47
Name: count, Length: 74, dtype: int64

Kolom: 'jenis_penempatan'  (3 distinct values)
jenis_penempatan
Magang       7277
Part-time    2931
Full-time    1792
Name: count, dtype: int64

Kolom: 'bi

In [18]:
pd.set_option("display.max_rows", None)

print(df["posisi"].value_counts(dropna=False))
print(df["bidang_studi_dicari"].value_counts(dropna=False))

posisi
Data Analyst                    1393
IT Support                       937
Quality Control Staff            406
Mechanical Engineer Intern       348
Business Analyst                 345
Digital Marketing Intern         341
Data Entry Operator              327
IT Staff                         326
Business Development Intern      304
Customer Service Intern          269
Marketing Staff                  233
Research Assistant               230
Marketing Intern                 228
UI/UX Designer                   210
Content Creator                  201
Frontend Developer               197
HSE Intern                       191
HR Intern                        178
Accounting Staff                 172
Environmental Analyst            147
Content Writer                   146
Store Supervisor Intern          145
Underwriting Intern              144
Visual Merchandiser Intern       144
Business Process Analyst         140
Supply Chain Analyst             136
Agricultural Engineer Intern   

In [19]:
def cek_variasi_penulisan(kolom):
    nilai_unik = df[kolom].dropna().unique()
    versi_bersih = {}
    for v in nilai_unik:
        key = str(v).strip().lower()
        versi_bersih.setdefault(key, []).append(v)
    
    print(f"\nKolom '{kolom}' — kandidat duplikat makna beda penulisan:")
    for key, variasi in versi_bersih.items():
        if len(variasi) > 1:
            print(f"  {variasi}")

cek_variasi_penulisan("posisi")
cek_variasi_penulisan("bidang_studi_dicari")


Kolom 'posisi' — kandidat duplikat makna beda penulisan:

Kolom 'bidang_studi_dicari' — kandidat duplikat makna beda penulisan:


## 2.4. Duplikasi

### Fully duplicated rows

-->> aman

In [20]:
jumlah_duplikat = df.duplicated().sum()
print(f"Duplikat baris penuh: {jumlah_duplikat}")

if jumlah_duplikat > 0:
    print(df[df.duplicated(keep=False)].sort_values(by=df.columns[0]))

Duplikat baris penuh: 0


### Primary key duplicated

-->> aman

In [21]:
pk = "id_tracking_company"  # ganti sesuai PK tabel ini

jumlah_dup_pk = df[pk].duplicated().sum()
print(f"Duplikat pada PK ('{pk}'): {jumlah_dup_pk}")

if jumlah_dup_pk > 0:
    print(df[df[pk].duplicated(keep=False)].sort_values(pk))

Duplikat pada PK ('id_tracking_company'): 0


### semanthic duplicate

In [22]:
# cek: apakah ada nama_perusahaan yang sama tapi id_company beda
cek_perusahaan = df.groupby("nama_perusahaan")["id_company"].nunique()
kandidat_1 = cek_perusahaan[cek_perusahaan > 1]
print("Nama perusahaan sama, tapi id_company berbeda:")
print(kandidat_1)

# cek sebaliknya: id_company sama tapi nama_perusahaan beda (ejaan tidak konsisten)
cek_id = df.groupby("id_company")["nama_perusahaan"].nunique()
kandidat_2 = cek_id[cek_id > 1]
print("\nid_company sama, tapi nama_perusahaan berbeda:")
print(kandidat_2)

Nama perusahaan sama, tapi id_company berbeda:
Series([], Name: id_company, dtype: int64)

id_company sama, tapi nama_perusahaan berbeda:
Series([], Name: nama_perusahaan, dtype: int64)


## Kategorikal

In [23]:
pd.set_option("display.max_rows", None)

### Value counts
-->> udah di cek di sebelumnya juga kok aman

### Variasi penulisan

In [24]:
def cek_variasi_penulisan(kolom):
    nilai_unik = df[kolom].dropna().unique()
    versi_bersih = {}
    for v in nilai_unik:
        key = str(v).strip().lower()
        versi_bersih.setdefault(key, []).append(v)

    hasil = {k: v for k, v in versi_bersih.items() if len(v) > 1}
    print(f"\nKolom '{kolom}' — kandidat variasi penulisan sama makna:")
    if hasil:
        for k, v in hasil.items():
            print(f"  {v}")
    else:
        print("  Tidak ditemukan.")

for kolom in kolom_kategorikal:
    cek_variasi_penulisan(kolom)

NameError: name 'kolom_kategorikal' is not defined

### Kemiripan

In [ ]:
from rapidfuzz import fuzz

def cek_typo_mirip(kolom, threshold=85, batas_unik=150):
    nilai_unik = df[kolom].dropna().unique().tolist()
    if len(nilai_unik) > batas_unik:
        print(f"\nKolom '{kolom}' dilewati untuk fuzzy check ({len(nilai_unik)} distinct, terlalu banyak)")
        return
    print(f"\nKolom '{kolom}' — pasangan mirip (similarity >= {threshold}):")
    ditemukan = False
    for i in range(len(nilai_unik)):
        for j in range(i+1, len(nilai_unik)):
            skor = fuzz.ratio(str(nilai_unik[i]).lower(), str(nilai_unik[j]).lower())
            if skor >= threshold:
                print(f"  '{nilai_unik[i]}'  <->  '{nilai_unik[j]}'   (skor: {skor})")
                ditemukan = True
    if not ditemukan:
        print("  Tidak ditemukan.")

for kolom in kolom_kategorikal:
    cek_typo_mirip(kolom)


Kolom 'id_tracking_company' dilewati untuk fuzzy check (12000 distinct, terlalu banyak)

Kolom 'id_talent_req' dilewati untuk fuzzy check (12000 distinct, terlalu banyak)

Kolom 'id_company' dilewati untuk fuzzy check (1495 distinct, terlalu banyak)

Kolom 'nama_perusahaan' dilewati untuk fuzzy check (1495 distinct, terlalu banyak)

Kolom 'posisi' — pasangan mirip (similarity >= 85):
  'Consultant Intern'  <->  'IT Consultant Intern'   (skor: 91.89189189189189)

Kolom 'jenis_penempatan' — pasangan mirip (similarity >= 85):
  Tidak ditemukan.

Kolom 'bidang_studi_dicari' — pasangan mirip (similarity >= 85):
  'Informatika, Pendidikan Teknik Informatika'  <->  'Sistem Informasi, Pendidikan Teknik Informatika'   (skor: 87.64044943820225)
  'Agroteknologi, Teknik Mesin'  <->  'Agroteknologi, Teknik Sipil'   (skor: 88.88888888888889)
  'Teknik Industri, Teknik Mesin'  <->  'Teknik Industri, Teknik Sipil'   (skor: 89.65517241379311)
  'Informatika, Desain Komunikasi Visual'  <->  'Sistem In

## 2.6. ID Format

### Penulisan ID

banyak banget ya -->> GA AMAN

In [ ]:
kolom_id_pattern = {
    "id_company": r"^C\d{3}$",
    "id_talent_req": r"^TR\d{3}$",
    "id_tracking_company": r"^TC\d{3}$"
    # sesuaikan pattern lain sesuai dokumentasi (SS, TS, dst)
}

for kolom, pattern in kolom_id_pattern.items():
    if kolom in df.columns:
        tidak_sesuai = df[~df[kolom].astype(str).str.match(pattern)]
        print(f"Kolom '{kolom}': {len(tidak_sesuai)} baris tidak sesuai pattern '{pattern}'")
        if len(tidak_sesuai) > 0:
            print(tidak_sesuai[kolom].unique()[:10])

Kolom 'id_company': 3796 baris tidak sesuai pattern '^C\d{3}$'
<StringArray>
['C1130', 'C1490', 'C1489', 'C1234', 'C1135', 'C1139', 'C1239', 'C1143',
 'C1152', 'C1185']
Length: 10, dtype: str
Kolom 'id_talent_req': 11001 baris tidak sesuai pattern '^TR\d{3}$'
<StringArray>
['TR1000', 'TR1001', 'TR1002', 'TR1003', 'TR1004', 'TR1005', 'TR1006',
 'TR1007', 'TR1008', 'TR1009']
Length: 10, dtype: str
Kolom 'id_tracking_company': 11001 baris tidak sesuai pattern '^TC\d{3}$'
<StringArray>
['TC1000', 'TC1001', 'TC1002', 'TC1003', 'TC1004', 'TC1005', 'TC1006',
 'TC1007', 'TC1008', 'TC1009']
Length: 10, dtype: str


### Leading Zero

-->> prefiks aman

In [ ]:
for kolom in kolom_id_pattern.keys():
    if kolom in df.columns:
        contoh = df[kolom].astype(str).unique()[:5]
        print(f"\nKolom '{kolom}' — contoh nilai: {contoh}")
        # cek apakah ada nilai yang cuma angka tanpa prefix huruf & tanpa leading zero
        murni_angka = df[kolom].astype(str).str.match(r"^\d+$")
        if murni_angka.sum() > 0:
            print(f"  [WARNING] {murni_angka.sum()} baris berupa angka murni (kemungkinan leading zero/prefix hilang)")


Kolom 'id_company' — contoh nilai: <StringArray>
['C171', 'C638', 'C900', 'C696', 'C506']
Length: 5, dtype: str

Kolom 'id_talent_req' — contoh nilai: <StringArray>
['TR001', 'TR002', 'TR003', 'TR004', 'TR005']
Length: 5, dtype: str

Kolom 'id_tracking_company' — contoh nilai: <StringArray>
['TC001', 'TC002', 'TC003', 'TC004', 'TC005']
Length: 5, dtype: str


### inkonsistensi panjang id

expected:
string id_company = 4
string id_talent_req = 5
string id_tracking_company = 5

-->> GA AMAN

In [ ]:
for kolom in kolom_id_pattern.keys():
    if kolom in df.columns:
        panjang = df[kolom].astype(str).str.len().value_counts()
        print(f"\nKolom '{kolom}' — distribusi panjang string:")
        print(panjang)


Kolom 'id_company' — distribusi panjang string:
id_company
4    8204
5    3796
Name: count, dtype: int64

Kolom 'id_talent_req' — distribusi panjang string:
id_talent_req
6    9000
7    2001
5     999
Name: count, dtype: int64

Kolom 'id_tracking_company' — distribusi panjang string:
id_tracking_company
6    9000
7    2001
5     999
Name: count, dtype: int64


In [ ]:
df["id_company"].astype(str).str.len().value_counts()
# cek sample di tiap grup panjang
print(df[df["id_company"].astype(str).str.len() == 4]["id_company"].sort_values().unique()[:5])
print(df[df["id_company"].astype(str).str.len() == 5]["id_company"].sort_values().unique()[:5])

<StringArray>
['C001', 'C002', 'C003', 'C004', 'C005']
Length: 5, dtype: str
<StringArray>
['C1000', 'C1001', 'C1002', 'C1003', 'C1004']
Length: 5, dtype: str


### Whitespace

-->> AMAN

In [ ]:
for kolom in kolom_id_pattern.keys():
    if kolom in df.columns:
        asli = df[kolom].astype(str)
        ada_whitespace = asli[asli != asli.str.strip()]
        print(f"\nKolom '{kolom}': {len(ada_whitespace)} baris dengan whitespace di ujung")
        if len(ada_whitespace) > 0:
            print(ada_whitespace.tolist()[:10])


Kolom 'id_company': 0 baris dengan whitespace di ujung

Kolom 'id_talent_req': 0 baris dengan whitespace di ujung

Kolom 'id_tracking_company': 0 baris dengan whitespace di ujung
